In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import time
import pickle
from copy import deepcopy
from collections import deque
import os

# --------------------------------------------------
#  CONFIGURATION (easy to tweak)
# --------------------------------------------------
ROWS, COLS, INAROW = 9, 10, 5           # board dimensions
NUM_ITERATIONS  = 50                     # how many self-play → train cycles
GAMES_PER_ITER  = 40                     # games generated per iteration
MCTS_SIMULATIONS = 400                   # simulations per move during self‑play
EVAL_SIMULATIONS = 800                   # simulations during evaluation matches
BATCH_SIZE       = 128                   # training batch size
EPOCHS_PER_ITER  = 10                    # training epochs per iteration
LR               = 1e-3                  # learning rate
WEIGHT_DECAY     = 1e-4
BUFFER_CAPACITY  = 200_000               # max positions in replay buffer
MIN_BUFFER_SIZE  = 5_000                 # don't train until we have this many
NUM_FILTERS      = 64                    # conv filters in the network
C_PUCT           = 2.0                   # PUCT exploration constant
DIRICHLET_ALPHA  = 0.3                   # Dirichlet noise parameter
DIRICHLET_EPS    = 0.25                  # weight of Dirichlet noise
TEMPERATURE_THRESH = 15                  # first N moves use temp=1, then greedy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Paths
MODEL_DIR = Path("../ml/models")
DATA_DIR  = Path("../data")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "selfplay").mkdir(parents=True, exist_ok=True)

In [ ]:
class ConnectFour:
    def __init__(self, rows=ROWS, cols=COLS, inarow=INAROW):
        self.rows = rows
        self.cols = cols
        self.inarow = inarow
        self.board = np.zeros((rows, cols), dtype=np.int8)
        self.current_player = 0
        self.done = False
        self.winner = None

    def clone(self):
        new = ConnectFour(self.rows, self.cols, self.inarow)
        new.board = self.board.copy()
        new.current_player = self.current_player
        new.done = self.done
        new.winner = self.winner
        return new

    def legal_moves(self):
        if self.done:
            return []
        return [c for c in range(self.cols) if self.board[0, c] == 0]

    def apply_move(self, col):
        if col not in self.legal_moves():
            return False
        for r in range(self.rows - 1, -1, -1):
            if self.board[r, col] == 0:
                self.board[r, col] = self.current_player + 1
                break
        self._check_termination(r, col)
        self.current_player = 1 - self.current_player
        return True

    def _check_termination(self, row, col):
        player = self.board[row, col]
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
        for dr, dc in directions:
            count = 1
            r, c = row + dr, col + dc
            while 0 <= r < self.rows and 0 <= c < self.cols and self.board[r, c] == player:
                count += 1
                r += dr
                c += dc
            r, c = row - dr, col - dc
            while 0 <= r < self.rows and 0 <= c < self.cols and self.board[r, c] == player:
                count += 1
                r -= dr
                c -= dc
            if count >= self.inarow:
                self.done = True
                self.winner = self.current_player  # player who just moved
                return
        if len(self.legal_moves()) == 0:
            self.done = True
            self.winner = None  # draw

    def display(self):
        symbols = {0: '.', 1: 'X', 2: 'O'}
        for r in range(self.rows):
            print(' '.join(symbols[self.board[r, c]] for c in range(self.cols)))
        print()